In [ ]:
!git clone https://github.com/hodakamori/ml-tutorial.git

In [ ]:
import pandas as pd

In [ ]:
lit_df = pd.read_excel("/content/ml-tutorial/data/polymer-degradation/literature.xlsx")
exp1_df = pd.read_excel("/content/ml-tutorial/data/polymer-degradation/exp1.xlsx")
exp2_df = pd.read_excel("/content/ml-tutorial/data/polymer-degradation/exp2.xlsx")

In [ ]:
lit_df.head()

In [ ]:
exp1_df.head()

In [ ]:
!pip install git+https://github.com/samoturk/mol2vec

In [ ]:
!pip install rdkit

In [ ]:
# https://github.com/samoturk/mol2vec/issues/14
def sentences2vec(sentences, model, unseen=None):
    """Generate vectors for each sentence (list) in a list of sentences. Vector is simply a
    sum of vectors for individual words.

    Parameters
    ----------
    sentences : list, array
        List with sentences
    model : word2vec.Word2Vec
        Gensim word2vec model
    unseen : None, str
        Keyword for unseen words. If None, those words are skipped.
        https://stats.stackexchange.com/questions/163005/how-to-set-the-dictionary-for-text-analysis-using-neural-networks/163032#163032

    Returns
    -------
    np.array
    """

    keys = set(model.wv.key_to_index)
    vec = []

    if unseen:
        unseen_vec = model.wv.get_vector(unseen)

    for sentence in sentences:
        if unseen:
            vec.append(
                sum(
                    [
                        (
                            model.wv.get_vector(y)
                            if y in set(sentence) & keys
                            else unseen_vec
                        )
                        for y in sentence
                    ]
                )
            )
        else:
            vec.append(
                sum(
                    [
                        model.wv.get_vector(y)
                        for y in sentence
                        if y in set(sentence) & keys
                    ]
                )
            )
    return np.array(vec)

In [ ]:
import numpy as np
from gensim.models import word2vec
from mol2vec.features import (
    DfVec,
    MolSentence,
    mol2alt_sentence,
)
from rdkit import Chem

smiles = "CCC"
model = word2vec.Word2Vec.load(
    "/content/ml-tutorial/data/polymer-degradation/model_300dim.pkl"
)
mols = [Chem.MolFromSmiles(smiles)]
mol_sentence = [MolSentence(mol2alt_sentence(mol, 1)) for mol in mols]
x_molvec = [DfVec(x) for x in sentences2vec(mol_sentence, model, unseen="UNK")]
x_molvec = np.array([x.vec for x in x_molvec])

In [ ]:
x_molvec

In [ ]:
def mol2vec(smiles):
    mols = [Chem.MolFromSmiles(smi) for smi in smiles]
    model = word2vec.Word2Vec.load(
        "/content/ml-tutorial/data/polymer-degradation/model_300dim.pkl"
    )
    mol_sentence = [MolSentence(mol2alt_sentence(mol, 1)) for mol in mols]
    x_molvec = [DfVec(x) for x in sentences2vec(mol_sentence, model, unseen="UNK")]
    x_molvec = np.array([x.vec for x in x_molvec])
    return x_molvec


X = mol2vec(["CCC", "CCO", "CCN"])
X.shape

In [ ]:
def create_pairwise_preferences(X, y):
    n_samples = X.shape[0]
    pairwise_X = []
    pairwise_y = []

    for i in range(n_samples):
        for j in range(i + 1, n_samples):
            if y[i] != y[j]:
                x_diff = X[i] - X[j]
                pairwise_X.append(x_diff)
                pairwise_y.append(1 if y[i] > y[j] else -1)
                pairwise_X.append(-x_diff)
                pairwise_y.append(-1 if y[i] > y[j] else 1)

    return np.array(pairwise_X), np.array(pairwise_y)

In [ ]:
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

In [ ]:
X_lit = mol2vec(lit_df["SMILES"])
y_lit = lit_df["Degradability"].values
pairwise_X_lit, pairwise_y_lit = create_pairwise_preferences(X_lit, y_lit)

X_exp1 = mol2vec(exp1_df["SMILES"])
y_exp1 = exp1_df["Degradability"].values
pairwise_X_exp1, pairwise_y_exp1 = create_pairwise_preferences(X_exp1, y_exp1)

X_exp2 = mol2vec(exp2_df["SMILES"])
y_exp2 = exp2_df["Degradability"].values
pairwise_X_exp2, pairwise_y_exp2 = create_pairwise_preferences(X_exp2, y_exp2)

In [ ]:
print(pairwise_X_lit.shape)
print(pairwise_X_exp1.shape)
print(pairwise_X_exp2.shape)

In [ ]:
X_pw_train = np.concatenate([pairwise_X_lit, pairwise_X_exp1])
y_pw_train = np.concatenate([pairwise_y_lit, pairwise_y_exp1])

In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC(C=1, dual=False, max_iter=10000)
svm.fit(X_pw_train, y_pw_train)

In [ ]:
y_train_pred = svm.predict(X_pw_train)

In [ ]:
def evaluate_ranking(X_test, y_test, svm):
    n_samples = X_test.shape[0]
    scores = np.zeros(n_samples)
    for i in range(n_samples):
        for j in range(n_samples):
            if i != j:
                diff = X_test[i] - X_test[j]
                score = svm.decision_function([diff])[0]
                scores[i] += score
    predicted_ranks = (-scores).argsort().argsort()
    true_ranks = (-y_test).argsort().argsort()

    return predicted_ranks, true_ranks

In [ ]:
pred_ranks_lit, true_ranks_lit = evaluate_ranking(X_lit, y_lit, svm)
pred_ranks_exp1, true_ranks_exp1 = evaluate_ranking(X_exp1, y_exp1, svm)
pred_ranks_exp2, true_ranks_exp2 = evaluate_ranking(X_exp2, y_exp2, svm)

In [ ]:
import matplotlib.pyplot as plt

plot_items = {
    "Literature (train)": {
        "True Rank": true_ranks_lit,
        "Predicted Rank": pred_ranks_lit,
        "logS": y_lit,
    },
    "Exp1 (train)": {
        "True Rank": true_ranks_exp1,
        "Predicted Rank": pred_ranks_exp1,
        "logS": y_exp1,
    },
    "Exp2 (test)": {
        "True Rank": true_ranks_exp2,
        "Predicted Rank": pred_ranks_exp2,
        "logS": y_exp2,
    },
}

fig = plt.figure()

for i, (key, item) in enumerate(plot_items.items()):
    ax = fig.add_subplot(1, 3, i + 1)
    df_plot = pd.DataFrame(item)
    n_samples = len(df_plot)
    df_plot = df_plot.sort_values("True Rank")

    pl = pd.plotting.parallel_coordinates(
        df_plot, "logS", colormap=plt.cm.coolwarm, ax=ax
    )

    ax.set_title(f"{key.capitalize()}")

    ax.set_ylabel("Rank")
    ax.invert_yaxis()
    ax.get_legend().remove()

    if i == 2:
        norm = plt.Normalize(df_plot["logS"].min(), df_plot["logS"].max())
        sm = plt.cm.ScalarMappable(cmap=plt.cm.coolwarm, norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax)
        cbar.set_label("logS")

fig.tight_layout()

In [ ]:
from scipy.stats import kendalltau, spearmanr
from sklearn.metrics import ndcg_score


def evaluate_ranking_metrics(true_ranks, predicted_ranks):
    metrics = {}

    # 1. Spearman's Rank Correlation Coefficient
    spearman_corr, _ = spearmanr(true_ranks, predicted_ranks)
    metrics["spearman_correlation"] = spearman_corr

    # 2. Kendall's Tau
    kendall_tau, _ = kendalltau(true_ranks, predicted_ranks)
    metrics["kendall_tau"] = kendall_tau

    # 3. Mean Reciprocal Rank (MRR)
    def mrr(true_ranks, predicted_ranks):
        sorted_indices = np.argsort(predicted_ranks)
        true_ranks_sorted = true_ranks[sorted_indices]
        rr = 1.0 / (np.argwhere(true_ranks_sorted == 0)[0][0] + 1)
        return rr

    metrics["mrr"] = mrr(true_ranks, predicted_ranks)

    # 4. Normalized Discounted Cumulative Gain (NDCG)
    # Note: sklearn's ndcg_score expects relevance scores, not ranks
    relevance_scores = len(true_ranks) - true_ranks  # Convert ranks to relevance scores
    metrics["ndcg"] = ndcg_score([relevance_scores], [predicted_ranks])

    # 5. Precision at k
    def precision_at_k(true_ranks, predicted_ranks, k):
        top_k_predicted = np.argsort(predicted_ranks)[:k]
        top_k_true = np.argsort(true_ranks)[:k]
        return len(set(top_k_predicted) & set(top_k_true)) / k

    metrics["precision_at_5"] = precision_at_k(true_ranks, predicted_ranks, 5)
    metrics["precision_at_10"] = precision_at_k(true_ranks, predicted_ranks, 10)

    return metrics

In [ ]:
metrics = evaluate_ranking_metrics(true_ranks_lit, pred_ranks_lit)
metrics

In [ ]:
metrics = evaluate_ranking_metrics(true_ranks_exp1, pred_ranks_exp1)
metrics

In [ ]:
metrics = evaluate_ranking_metrics(true_ranks_exp2, pred_ranks_exp2)
metrics